In [79]:
import numpy as np

In [80]:
X_train = np.load('X_train.npy')
X_val = np.load('X_test.npy')
y_train = np.load('y_train.npy')
y_val = np.load('y_test.npy')

In [81]:
def get_location_category(category, X, y):
    i = 0
    x_loc = []
    y_loc = []
    for feature in X:
        if(y[i] == category):
            location = X[i, -2:]
            x_i = location[0]
            y_i = location[1]
            x_loc.append(x_i)
            y_loc.append(y_i)
        i = i+1
    x_loc = np.array(x_loc)
    y_loc = np.array(y_loc)
    locs = np.vstack((x_loc, y_loc)).T
    return locs

In [82]:
locs0 = get_location_category(0, X_train, y_train)
locs1 = get_location_category(1, X_train, y_train)
locs2 = get_location_category(2, X_train, y_train)
locs3 = get_location_category(3, X_train, y_train)
locs4 = get_location_category(4, X_train, y_train)
locs5 = get_location_category(5, X_train, y_train)
locs6 = get_location_category(6, X_train, y_train)
locs7 = get_location_category(7, X_train, y_train)
locs8 = get_location_category(8, X_train, y_train)
locs9 = get_location_category(9, X_train, y_train)

# Feature Engineering 1: Use KDEs for each category of crime fit to the location and append these log probabilities for each class as a feature

In [141]:
from sklearn.preprocessing import StandardScaler

In [144]:
scaler = StandardScaler()
locs0 = scaler.fit_transform(locs0)
locs1 = scaler.fit_transform(locs1)
locs2 = scaler.fit_transform(locs2)
locs3 = scaler.fit_transform(locs3)
locs4 = scaler.fit_transform(locs4)
locs5 = scaler.fit_transform(locs5)
locs6 = scaler.fit_transform(locs6)
locs7 = scaler.fit_transform(locs7)
locs8 = scaler.fit_transform(locs8)
locs9 = scaler.fit_transform(locs9)

In [217]:
from sklearn.neighbors import KernelDensity
def fit_kernel_density(locs_train):
    kde = KernelDensity(
        kernel = "gaussian",
        bandwidth = 0.1,
        algorithm = "ball_tree",
    ).fit(locs_train)
    return kde

In [218]:
kde0 = fit_kernel_density(locs0)
kde1 = fit_kernel_density(locs1)
kde2 = fit_kernel_density(locs2)
kde3 = fit_kernel_density(locs3)
kde4 = fit_kernel_density(locs4)
kde5 = fit_kernel_density(locs5)
kde6 = fit_kernel_density(locs6)
kde7 = fit_kernel_density(locs7)
kde8 = fit_kernel_density(locs8)
kde9 = fit_kernel_density(locs9)
kdes = [kde0, kde1, kde2, kde3, kde4, kde5, kde6, kde7, kde8, kde9]

In [220]:
def get_location(X):
    return np.array(X[:, -2:])

In [221]:
from tqdm.auto import tqdm
def kde_log_features(X_locs, kdes):
    num_cats = len(kdes)
    feats = []
    for c in tqdm(range(num_cats)):
        log_dens = kdes[c].score_samples(X_locs)
        feats.append(log_dens)
    return np.vstack(feats).T

In [222]:
X_train_loc = get_location(X_train)
X_val_loc = get_location(X_val)
f_train_kde = kde_log_features(X_train_loc, kdes)
f_val_kde = kde_log_features(X_val_loc, kdes)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [224]:
X_train_kde = np.hstack((X_train, f_train_kde))
X_val_kde = np.hstack((X_val, f_val_kde))

In [226]:
np.save('X_train_kde.npy', X_train_kde)
np.save('X_test_kde.npy', X_val_kde)

# Feature Engineering 2: Use kmeans clustering to make "hotspots" for each category of crime in training data, compute distance to nearest hotspot for each category and add append this as feature

In [156]:
locs0 = get_location_category(0, X_train, y_train)
locs1 = get_location_category(1, X_train, y_train)
locs2 = get_location_category(2, X_train, y_train)
locs3 = get_location_category(3, X_train, y_train)
locs4 = get_location_category(4, X_train, y_train)
locs5 = get_location_category(5, X_train, y_train)
locs6 = get_location_category(6, X_train, y_train)
locs7 = get_location_category(7, X_train, y_train)
locs8 = get_location_category(8, X_train, y_train)
locs9 = get_location_category(9, X_train, y_train)

In [97]:
from sklearn.cluster import KMeans

In [131]:
def make_clusters(locs_train):
    km = KMeans(
        n_clusters = 120,
        n_init = "auto",
        random_state = 42,
    )
    return km.fit(locs_train)

In [132]:
km0 = make_clusters(locs0)
km1 = make_clusters(locs1)
km2 = make_clusters(locs2)
km3 = make_clusters(locs3)
km4 = make_clusters(locs4)
km5 = make_clusters(locs5)
km6 = make_clusters(locs6)
km7 = make_clusters(locs7)
km8 = make_clusters(locs8)
km9 = make_clusters(locs9)
kms = [km0, km1, km2, km3, km4, km5, km6, km7, km8, km9]

In [134]:
from sklearn.metrics.pairwise import pairwise_distances
def km_dist(X, kms):
    feats = []
    num_cats = len(kms)
    for c in tqdm(range(num_cats)):
        km = kms[c]
        dists = pairwise_distances(X, km.cluster_centers_, metric = "haversine")
        min_dist = dists.min(axis=1)
        feats.append(min_dist)
    return np.vstack(feats).T

In [135]:
f_train_km = km_dist(X_train_loc, kms)
f_val_km = km_dist(X_val_loc, kms)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [136]:
f_train_km_even = km_dist(X_train_loc_even, kms_even)
f_val_km_even = km_dist(X_val_loc_even, kms_even)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [137]:
X_train_kms = np.hstack((X_train, f_train_km))
X_val_kms = np.hstack((X_val, f_val_km))

In [140]:
np.save('X_train_kms.npy', X_train_kms)
np.save('X_test_kms.npy', X_val_kms)